# Clustering Algorithms on Topology with topologic_fast

This notebook demonstrates clustering algorithms applied to topological data structures using `topologic_fast`, a high-performance Rust-based topology library.

## Overview

Clustering is an unsupervised learning technique that groups similar data points together. When applied to topological and geometric data, clustering can help identify:
- Spatial groupings of vertices
- Similar geometric features
- Regions in architectural floor plans

In this notebook, we'll:
1. Create geometric shapes using topologic_fast
2. Extract vertex data from topologies
3. Apply clustering algorithms (K-Means, DBSCAN) using scikit-learn
4. Visualize the results using Plotly

In [ ]:
# Import required libraries
import topologic_fast as tf
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans, DBSCAN
import warnings
warnings.filterwarnings('ignore')

## 1. Create Sample Geometric Shapes

We'll create several rectangular faces at different locations and extract their vertices for clustering.

In [ ]:
# Create three rectangular faces at different positions
r1 = tf.Face.Rectangle(x=0, y=0, z=0, width=2, length=2)
r2 = tf.Face.Rectangle(x=5, y=0, z=0, width=5, length=1)
r3 = tf.Face.Rectangle(x=1.5, y=-2.2, z=0, width=3, length=3)

# Create connector shapes
r4 = tf.Face.Rectangle(x=0.75, y=-0.5, z=0, width=0.2, length=0.75)
r5 = tf.Face.Rectangle(x=2.75, y=-0.5, z=0, width=0.2, length=0.75)

print(f"Created 5 rectangular faces")
print(f"r1: {r1}")
print(f"r2: {r2}")
print(f"r3: {r3}")

## 2. Extract Vertices from Shapes

We'll extract the vertices from the rectangles and use them as data points for clustering.

In [ ]:
def extract_vertices_as_array(faces):
    """Extract vertices from a list of faces and return as numpy array."""
    points = []
    for face in faces:
        vertices = face.Vertices()
        for v in vertices:
            points.append(v.Coordinates())
    return np.array(points)

# Get vertices from main rectangles
rectangles = [r1, r2, r3]
points = extract_vertices_as_array(rectangles)

print(f"Extracted {len(points)} vertices from {len(rectangles)} rectangles")
print(f"Points shape: {points.shape}")
print(f"\nSample points:")
print(points[:5])

## 3. Visualize the Points

Before clustering, let's visualize the vertex positions.

In [ ]:
def plot_points_2d(points, colors=None, title="Points", size=12):
    """Plot 2D points using Plotly."""
    fig = go.Figure()
    
    if colors is None:
        colors = ['blue'] * len(points)
    
    fig.add_trace(go.Scatter(
        x=points[:, 0],
        y=points[:, 1],
        mode='markers',
        marker=dict(size=size, color=colors),
        text=[f"({p[0]:.2f}, {p[1]:.2f})" for p in points],
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title=title,
        xaxis_title="X",
        yaxis_title="Y",
        width=700,
        height=500,
        plot_bgcolor='white',
        showlegend=False
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', scaleanchor='x')
    
    return fig

fig = plot_points_2d(points, title="Vertices from Rectangular Faces")
fig.show()

## 4. K-Means Clustering

K-Means partitions data into K clusters by minimizing the within-cluster sum of squares.

**Note:** topologicpy provides `Cluster.KMeans()` directly. In topologic_fast, we use scikit-learn for clustering since the focus is on high-performance topology operations. This demonstrates how topologic_fast integrates seamlessly with the Python ML ecosystem.

In [ ]:
# Apply K-Means clustering with K=3 (since we have 3 rectangles)
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(points[:, :2])  # Use X, Y coordinates

# Get cluster centers
centroids = kmeans.cluster_centers_

print(f"K-Means clustering with k={k}")
print(f"Cluster labels: {cluster_labels}")
print(f"\nCluster centroids:")
for i, c in enumerate(centroids):
    print(f"  Cluster {i}: ({c[0]:.3f}, {c[1]:.3f})")

In [ ]:
def plot_clustered_points_2d(points, labels, centroids=None, title="Clustered Points"):
    """Plot clustered 2D points with different colors per cluster."""
    colors = ['red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta', 'yellow']
    
    fig = go.Figure()
    
    unique_labels = np.unique(labels)
    for label in unique_labels:
        mask = labels == label
        cluster_points = points[mask]
        color = colors[label % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=cluster_points[:, 0],
            y=cluster_points[:, 1],
            mode='markers',
            marker=dict(size=14, color=color),
            name=f'Cluster {label}',
            text=[f"Cluster {label}: ({p[0]:.2f}, {p[1]:.2f})" for p in cluster_points],
            hoverinfo='text'
        ))
    
    # Plot centroids
    if centroids is not None:
        fig.add_trace(go.Scatter(
            x=centroids[:, 0],
            y=centroids[:, 1],
            mode='markers',
            marker=dict(size=20, color='black', symbol='x'),
            name='Centroids'
        ))
    
    fig.update_layout(
        title=title,
        xaxis_title="X",
        yaxis_title="Y",
        width=700,
        height=500,
        plot_bgcolor='white',
        legend=dict(x=1.02, y=1)
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', scaleanchor='x')
    
    return fig

fig = plot_clustered_points_2d(points, cluster_labels, centroids, 
                               title=f"K-Means Clustering (k={k})")
fig.show()

## 5. DBSCAN Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) is a density-based clustering algorithm that:
- Does not require specifying the number of clusters beforehand
- Can find arbitrarily shaped clusters
- Identifies noise points

Key parameters:
- `epsilon`: Maximum distance between points in the same neighborhood
- `min_samples`: Minimum number of points to form a dense region

In [ ]:
# Apply DBSCAN clustering
epsilon = 0.6
min_samples = 3

dbscan = DBSCAN(eps=epsilon, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(points[:, :2])

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f"DBSCAN clustering with epsilon={epsilon}, min_samples={min_samples}")
print(f"Number of clusters found: {n_clusters}")
print(f"Number of noise points: {n_noise}")
print(f"Cluster labels: {dbscan_labels}")

In [ ]:
def plot_dbscan_clusters(points, labels, title="DBSCAN Clustering"):
    """Plot DBSCAN clustered points, with noise points in gray."""
    colors = ['red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta', 'yellow']
    
    fig = go.Figure()
    
    unique_labels = np.unique(labels)
    for label in unique_labels:
        mask = labels == label
        cluster_points = points[mask]
        
        if label == -1:
            # Noise points
            fig.add_trace(go.Scatter(
                x=cluster_points[:, 0],
                y=cluster_points[:, 1],
                mode='markers',
                marker=dict(size=10, color='gray', opacity=0.5),
                name='Noise',
                text=[f"Noise: ({p[0]:.2f}, {p[1]:.2f})" for p in cluster_points],
                hoverinfo='text'
            ))
        else:
            color = colors[label % len(colors)]
            fig.add_trace(go.Scatter(
                x=cluster_points[:, 0],
                y=cluster_points[:, 1],
                mode='markers',
                marker=dict(size=14, color=color),
                name=f'Cluster {label}',
                text=[f"Cluster {label}: ({p[0]:.2f}, {p[1]:.2f})" for p in cluster_points],
                hoverinfo='text'
            ))
    
    fig.update_layout(
        title=title,
        xaxis_title="X",
        yaxis_title="Y",
        width=700,
        height=500,
        plot_bgcolor='white',
        legend=dict(x=1.02, y=1)
    )
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
    fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', scaleanchor='x')
    
    return fig

fig = plot_dbscan_clusters(points, dbscan_labels, 
                           title=f"DBSCAN Clustering (eps={epsilon}, min_samples={min_samples})")
fig.show()

## 6. Clustering on More Complex Topology

Let's create a more complex topology using spirals and apply clustering to demonstrate the algorithms on irregular point distributions.

In [ ]:
# Create three spiral wires
spiral1 = tf.Wire.Spiral(radius_a=1, radius_b=5, height=10, turns=2, sides=36)
spiral2 = tf.Wire.Spiral(radius_a=1, radius_b=5, height=10, turns=2, sides=36)
spiral3 = tf.Wire.Spiral(radius_a=1, radius_b=5, height=10, turns=2, sides=36)

# Get vertices from spirals
# For spiral1, use as is
verts1 = spiral1.Vertices()

# For spiral2 and spiral3, we need to rotate them
# Extract coordinates and manually rotate since Topology.RotateWire exists

def get_vertex_coords(wire):
    """Extract vertex coordinates from a wire."""
    return np.array([v.Coordinates() for v in wire.Vertices()])

def rotate_points(points, angle_degrees, axis='z'):
    """Rotate points around an axis."""
    angle = np.radians(angle_degrees)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    
    if axis == 'z':
        rotation_matrix = np.array([
            [cos_a, -sin_a, 0],
            [sin_a, cos_a, 0],
            [0, 0, 1]
        ])
    return points @ rotation_matrix.T

# Get and rotate spiral points
pts1 = get_vertex_coords(spiral1)
pts2 = rotate_points(get_vertex_coords(spiral1), 120)
pts3 = rotate_points(get_vertex_coords(spiral1), 240)

# Combine all spiral points
spiral_points = np.vstack([pts1, pts2, pts3])

print(f"Created 3 spirals with total {len(spiral_points)} vertices")

In [ ]:
# Visualize the spiral points in 3D
fig = go.Figure(data=[go.Scatter3d(
    x=spiral_points[:, 0],
    y=spiral_points[:, 1],
    z=spiral_points[:, 2],
    mode='markers',
    marker=dict(size=4, color='blue')
)])

fig.update_layout(
    title="Spiral Vertex Points (3D)",
    width=700,
    height=600,
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)
fig.show()

In [ ]:
# Calculate inter-point distance for DBSCAN epsilon parameter
# Use the distance between consecutive points on the spiral
inter_point_dist = np.linalg.norm(pts1[1] - pts1[0])
print(f"Inter-point distance: {inter_point_dist:.4f}")

# Apply DBSCAN with appropriate epsilon
epsilon_spiral = inter_point_dist * 1.1
dbscan_spiral = DBSCAN(eps=epsilon_spiral, min_samples=2)
spiral_labels = dbscan_spiral.fit_predict(spiral_points)

n_clusters_spiral = len(set(spiral_labels)) - (1 if -1 in spiral_labels else 0)
print(f"\nDBSCAN on spirals:")
print(f"  Epsilon: {epsilon_spiral:.4f}")
print(f"  Clusters found: {n_clusters_spiral}")

In [ ]:
# Visualize clustered spiral points in 3D
colors_map = ['red', 'green', 'blue', 'orange', 'purple', 'cyan']
point_colors = [colors_map[l % len(colors_map)] if l >= 0 else 'gray' for l in spiral_labels]

fig = go.Figure(data=[go.Scatter3d(
    x=spiral_points[:, 0],
    y=spiral_points[:, 1],
    z=spiral_points[:, 2],
    mode='markers',
    marker=dict(size=5, color=point_colors)
)])

fig.update_layout(
    title=f"DBSCAN Clustering on Spirals ({n_clusters_spiral} clusters)",
    width=700,
    height=600,
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)
fig.show()

## 7. Graph-Based Clustering Approach

topologic_fast provides graph utilities that can be used for graph-based clustering. We can create a graph from topology and use graph properties for analysis.

In [ ]:
# Create a cell complex (3D grid)
cell_complex = tf.CellComplex.Box(0, 0, 0, 3, 3, 3)
print(f"Cell complex: {cell_complex}")

# Create a graph from the cell complex
# This creates a dual graph where each cell becomes a vertex
graph = tf.Graph.ByTopology(cell_complex, direct=True)

print(f"\nGraph from cell complex:")
print(f"  Vertices (cells): {graph.Order()}")
print(f"  Edges (adjacencies): {graph.Size()}")
print(f"  Density: {graph.Density():.4f}")

In [ ]:
# Get graph vertices and their positions
graph_vertices = graph.Vertices()
graph_positions = np.array([v.Coordinates() for v in graph_vertices])

print(f"Graph vertex positions:")
for i, pos in enumerate(graph_positions):
    print(f"  Node {i}: ({pos[0]:.2f}, {pos[1]:.2f}, {pos[2]:.2f})")

# Get adjacency information
adj_matrix = graph.AdjacencyMatrix()
print(f"\nAdjacency matrix shape: {len(adj_matrix)}x{len(adj_matrix[0])}")

In [ ]:
# Visualize the graph
edges = graph.Edges()

# Create edge lines for visualization
edge_x = []
edge_y = []
edge_z = []

for edge in edges:
    start_v, end_v = edge.Vertices()
    start = start_v.Coordinates()
    end = end_v.Coordinates()
    edge_x.extend([start[0], end[0], None])
    edge_y.extend([start[1], end[1], None])
    edge_z.extend([start[2], end[2], None])

fig = go.Figure()

# Add edges
fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='gray', width=2),
    name='Edges'
))

# Add vertices
fig.add_trace(go.Scatter3d(
    x=graph_positions[:, 0],
    y=graph_positions[:, 1],
    z=graph_positions[:, 2],
    mode='markers',
    marker=dict(size=10, color='red'),
    name='Vertices',
    text=[f"Node {i}" for i in range(len(graph_positions))],
    hoverinfo='text'
))

fig.update_layout(
    title="Graph from Cell Complex (Dual Graph)",
    width=700,
    height=600,
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    )
)
fig.show()

## 8. Summary

In this notebook, we demonstrated:

1. **Creating topology with topologic_fast**: Used `Face.Rectangle()` and `Wire.Spiral()` to create geometric shapes

2. **Extracting vertex data**: Retrieved coordinates from topological entities for use in clustering

3. **K-Means clustering**: Applied K-Means to partition vertices into K groups based on spatial proximity

4. **DBSCAN clustering**: Used density-based clustering to find clusters without specifying K, with noise detection

5. **Graph-based analysis**: Created graphs from cell complexes for structural analysis

### Key Differences from topologicpy

| Feature | topologicpy | topologic_fast |
|---------|-------------|----------------|
| Cluster.KMeans() | Built-in | Use scikit-learn |
| Cluster.DBSCAN() | Built-in | Use scikit-learn |
| Topology.Show() | Built-in | Use Plotly directly |
| Performance | Python | Rust (faster) |

topologic_fast focuses on high-performance topology operations, while clustering algorithms can be easily integrated using scikit-learn or other Python ML libraries.

In [ ]:
# Clean up topology store
tf.clear_store()
print("Topology store cleared.")